# OPEN WORK ORDERS ⚠

In [0]:
%sql
CREATE OR REPLACE TABLE fm_prj1.gold.open_work_orders AS
SELECT *
FROM fm_prj1.silver.work_order
WHERE status = 'Open';

# BUSINESS LAYER

## SLA SUMMARY

In [0]:
%sql
CREATE OR REPLACE TABLE fm_prj1.gold.gold_sla_summary AS
SELECT
  YEAR(open_date) AS open_year,
  MONTH(open_date) AS open_month,
  priority,

  -- Total number of closed work orders per year, month, and priority
  COUNT(*) AS total_work_orders,

  -- Number of work orders which met SLA
  SUM(sla_met) AS sla_met_count,
  -- Number of work orders which did not meet SLA
  COUNT(*) - SUM(sla_met) AS sla_breached_count,

  -- SLA Compliance Percentage
  ROUND(SUM(sla_met) / COUNT(*) * 100, 2) AS sla_compliance_percentage,
  -- SLA Breach Percentage
  ROUND((COUNT(*) - SUM(sla_met)) / COUNT(*) * 100, 2) AS sla_breach_percentage,

  -- Average response hours for closed work orders
  ROUND(AVG(response_hours), 2) AS avg_response_hours,
  -- Average close hours for closed work orders
  ROUND(AVG(close_hours), 2) AS avg_close_hours,

  -- Average breach hours for work orders that breached SLA
  ROUND(AVG(sla_breach_hours), 2) AS avg_sla_breach_hours

FROM fm_prj1.silver.work_order
WHERE status = 'Closed'

GROUP BY
  YEAR(open_date),
  MONTH(open_date),
  priority

ORDER BY
  open_year,
  open_month,
  priority;

## TECHNICIAN PERFORMANCE

## ASSET PERFORMANCE

# SILVER TO GOLD DELTA TABLE

In [0]:
%sql
CREATE OR REPLACE TABLE fm_prj1.gold.dim_asset AS
SELECT * FROM fm_prj1.silver.asset;

CREATE OR REPLACE TABLE fm_prj1.gold.dim_technician AS
SELECT * FROM fm_prj1.silver.technician;

CREATE OR REPLACE TABLE fm_prj1.gold.fact_work_order AS
SELECT * FROM fm_prj1.silver.work_order;

In [0]:
# print("Writing to gold delta table...")
# (
#     df.write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable("fm_prj1.gold.fact_work_order")
# )
# print("Succesfully completed writing to gold delta table")